# CFPB Company Identity — canonical name mapping

**Goal:** a conservative canonical `Company` mapping so the same lender is not counted as
several lenders because of capitalization, punctuation, DBA/FKA naming, or known
company-name changes.

**Constraint that drives every decision here:** *false merges are worse than missed
merges.* Merging two real lenders silently misattributes thousands of complaints;
leaving one lender split across two keys inflates the lender count by one and is
recoverable. So every rule below is one-sided: a merge requires positive evidence, and
similarity alone is never sufficient.

**Scope:** data cleaning only. No scoring, no NLP, no clustering, no product changes.
The raw CSV is read-only.

**Prior art:** `pipeline/normalize.py` from commit `53f6208` (deleted from the tree) is a
tested normalizer with a longer suffix list and a generic-word stripper. It is *not*
restored here, for two reasons: it was written against the 5.46 GB national bulk file
rather than this 10 MB extract, and it merges more aggressively than this dataset can
support without evidence.

In [1]:
import difflib
import re
import unicodedata
from collections import Counter, defaultdict

import pandas as pd

REPO = "/Users/mihirkale/cdc"
RAW = f"{REPO}/data/raw/paydayComplaints.csv"
OUT = f"{REPO}/data/processed/company_aliases.csv"
NARR = f"{REPO}/data/raw/narratives_cache/matched_narratives.csv"

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 110)

## 1. The starting point

In [2]:
df = pd.read_csv(RAW, usecols=["Complaint ID", "Company", "Sub-product", "State", "Date received"])
counts = df["Company"].value_counts()
print(f"complaints        : {len(df):,}")
print(f"unique Company    : {len(counts):,}")
print(f"median per company: {counts.median():.0f}")
print(f"\ncompanies with 1 complaint: {(counts == 1).sum():,} ({(counts == 1).mean() * 100:.1f}%)")
print(f"companies with >=10      : {(counts >= 10).sum():,}")
counts.head(12).to_frame("complaints")

complaints        : 38,375
unique Company    : 1,162
median per company: 2

companies with 1 complaint: 434 (37.3%)
companies with >=10      : 285


,complaints
Company,
"Affirm Holdings, Inc",3257
OneMain Finance Corporation,2281
CCF Intermediate Holdings LLC,1149
"Paramount GR Holdings, LLC",1132
"ENOVA INTERNATIONAL, INC.",1099
Klarna AB,805
TRUIST FINANCIAL CORPORATION,597
SYNCHRONY FINANCIAL,595
"Upstart Holdings, Inc.",578


## 2. Conservative normalization primitives

Three separate keys, each stricter than the last. Only the first two may drive a merge.

| Key | Definition | Used to |
| --- | --- | --- |
| `surface` | the raw string | identity of a filer's own spelling |
| `alnum` | NFKD-folded, ASCII, upper, non-alphanumeric collapsed to single spaces | **may merge** — catches case, punctuation, whitespace |
| `stem` | `alnum` minus legal-entity suffixes | **may merge** — catches `Inc.` / `LLC` variation |
| DBA/FKA strip | removes a trailing `d/b/a` brand or a `(F/K/A ...)` parenthetical | **relabel only** — a brand is not a second filer |

Deliberately *not* applied: stripping generic corporate words (`HOLDINGS`, `FINANCIAL`,
`SERVICES`, `GROUP`). The recovered `config.py` documented the same caution — dropping
`FINANCIAL` collapses genuinely distinct filers. The risk/reward is checked in section 5
instead of being assumed.

In [3]:
SUFFIXES = {
    "INC", "INCORPORATED", "LLC", "L L C", "LP", "L P", "LLP", "L L P",
    "CORP", "CORPORATION", "CO", "COMPANY", "LTD", "LIMITED", "PLC", "PC", "PA",
}

# A parenthetical that names a predecessor, e.g. "(F/K/A Ace Cash Express)".
# The slash is optional so that "F/K/A", "FKA" and "F K A" all match.
HISTORY_PAREN = re.compile(
    r"\s*\([^)]*\b(?:f\s*/?\s*k\s*/?\s*a|d\s*/?\s*b\s*/?\s*a|a\s*/?\s*k\s*/?\s*a"
    r"|n\s*/?\s*k\s*/?\s*a|formerly|previously|now known)\b[^)]*\)",
    re.I,
)
# A trailing "d/b/a Brand" or "dba Brand", e.g. "Ningo Lending LLC dba Spotloan".
TRAILING_MARKER = re.compile(
    r"\s*,?\s*\b(?:d\s*/?\s*b\s*/?\s*a|a\s*/?\s*k\s*/?\s*a"
    r"|f\s*/?\s*k\s*/?\s*a|n\s*/?\s*k\s*/?\s*a)\b.*$",
    re.I,
)
# Same marker without the trailing ".*$", so the brand name can be recovered by splitting.
MARKER_ONLY = re.compile(
    r"\b(?:d\s*/?\s*b\s*/?\s*a|a\s*/?\s*k\s*/?\s*a"
    r"|f\s*/?\s*k\s*/?\s*a|n\s*/?\s*k\s*/?\s*a)\b",
    re.I,
)


def alnum(s):
    t = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", re.sub(r"[^A-Za-z0-9]+", " ", t).upper()).strip()


def stem(s):
    toks = alnum(s).split()
    kept = [t for t in toks if t not in SUFFIXES]
    return " ".join(kept) if kept else " ".join(toks)


def strip_marker(s):
    return TRAILING_MARKER.sub("", HISTORY_PAREN.sub(" ", str(s))).strip(" ,;()-")


names = list(counts.index)
alnum_map = defaultdict(list)
stem_map = defaultdict(list)
for n in names:
    alnum_map[alnum(n)].append(n)
    stem_map[stem(n)].append(n)

print(f"distinct alnum keys : {len(alnum_map):,}  (from {len(names):,} surface names)")
print(f"distinct stem keys  : {len(stem_map):,}")
print(f"names carrying a DBA/AKA/FKA marker : "
      f"{sum(bool(TRAILING_MARKER.search(n) or HISTORY_PAREN.search(n)) for n in names):,}")

distinct alnum keys : 1,161  (from 1,162 surface names)
distinct stem keys  : 1,160
names carrying a DBA/AKA/FKA marker : 33


## 3. `alnum` collisions — case, punctuation, whitespace

These are the same string modulo case/punctuation/whitespace. Merging is safe by
construction: the character content is identical.

In [4]:
def collision_groups(mapping):
    return {k: v for k, v in mapping.items() if len(v) > 1}


alnum_groups = collision_groups(alnum_map)
stem_groups = collision_groups(stem_map)

print(f"=== alnum collision groups: {len(alnum_groups)} ===")
for k, v in sorted(alnum_groups.items(), key=lambda kv: -sum(counts[z] for z in kv[1])):
    print(f"\n  key {k!r}")
    for z in sorted(v, key=lambda z: -counts[z]):
        print(f"      {counts[z]:>6}  {z!r}")

print(f"\n=== stem collision groups: {len(stem_groups)} ===")
for k, v in sorted(stem_groups.items(), key=lambda kv: -sum(counts[z] for z in kv[1])):
    print(f"\n  stem {k!r}")
    for z in sorted(v, key=lambda z: -counts[z]):
        print(f"      {counts[z]:>6}  {z!r}")

=== alnum collision groups: 1 ===

  key 'FIRST TECHNOLOGY FEDERAL CREDIT UNION'
          51  'FIRST TECHNOLOGY FEDERAL CREDIT UNION'
          24  'First Technology Federal Credit Union'

=== stem collision groups: 2 ===

  stem 'FIRST TECHNOLOGY FEDERAL CREDIT UNION'
          51  'FIRST TECHNOLOGY FEDERAL CREDIT UNION'
          24  'First Technology Federal Credit Union'

  stem 'TIME INVESTMENT'
           9  'Time Investment Corporation'
           8  'Time Investment Company, Inc.'


`alnum` collapses the `STATE EMPLOYEES' CREDIT UNION` typographic-apostrophe variant onto
its plain form automatically. The other two groups need judgement, so they are examined
below rather than merged on sight.

In [5]:
# "Time Investment Corporation" (GA, NC) vs "Time Investment Company, Inc."
# (AR, AZ, CA, FL, GA, MO, NM): same stem, but they share only GA and both are active
# in the same period. Consistent with two related-but-distinct registrations.
ti = df[df["Company"].str.contains("Time Investment", case=False)]
ti.groupby("Company").agg(
    complaints=("Complaint ID", "size"),
    states=("State", lambda s: ", ".join(sorted(set(s.dropna())))),
    first=("Date received", "min"),
    last=("Date received", "max"),
).to_string()

'                               complaints                      states                     first                      last\nCompany                                                                                                                  \nTime Investment Company, Inc.           8  AR, AZ, CA, FL, GA, MO, NM  2024-03-13T19:50:35.000Z  2026-08-18T00:48:14.000Z\nTime Investment Corporation             9                      GA, NC  2023-08-29T18:08:51.000Z  2026-09-18T17:31:49.000Z'

## 4. DBA / AKA / FKA names

A `d/b/a` brand is not a second filer — it is the same filer trading under a brand. So the
canonical label should be the legal entity. The question that matters for *merging* is
whether stripping the brand makes two distinct rows collide.

In [6]:
marked = [n for n in names if TRAILING_MARKER.search(n) or HISTORY_PAREN.search(n)]
print(f"names with a DBA/AKA/FKA marker: {len(marked)}\n")

stripped_map = defaultdict(list)
for n in marked:
    stripped_map[stem(strip_marker(n))].append(n)
collide = {k: v for k, v in stripped_map.items() if len(v) > 1}

print(f"marker-stripped stems shared by >1 name: {len(collide)}")
for k, v in collide.items():
    print(f"   stem {k!r} <- {v}")

# The decisive test: does any DBA brand also exist as its own standalone row?
# If it did, stripping the marker would merge two filers and must be avoided.
brand_hits = []
for n in marked:
    m = re.split(r"\b(?:d\s*/\s*b\s*/\s*a|a\s*/\s*k\s*/\s*a|f\s*/\s*k\s*/\s*a)\b", n, flags=re.I)
    if len(m) > 1:
        brand = m[-1].strip(" ,;()-")
        hits = [y for y in names if y != n and alnum(y) == alnum(brand)]
        if hits:
            brand_hits.append((n, brand, hits))

print(f"\nDBA brands that also exist as a standalone row: {len(brand_hits)}")
for n, b, h in brand_hits:
    print(f"   {n!r} -> brand {b!r} -> standalone {h}")
print("\n=> No brand collides. Stripping the marker relabels 33 filers but merges none of them.")

names with a DBA/AKA/FKA marker: 33

marker-stripped stems shared by >1 name: 0

DBA brands that also exist as a standalone row: 0

=> No brand collides. Stripping the marker relabels 33 filers but merges none of them.


In [7]:
print("=== the marker-bearing names, as they will be relabelled ===")
rows = []
for n in sorted(marked, key=lambda z: -counts[z]):
    marker = "FKA" if HISTORY_PAREN.search(n) else ("DBA" if TRAILING_MARKER.search(n) else "")
    hist = HISTORY_PAREN.findall(n)
    rows.append({"complaints": counts[n], "original": n, "type": marker,
                 "canonical_label": strip_marker(n),
                 "predecessor_named": "; ".join(hist) if hist else ""})
pd.DataFrame(rows)

=== the marker-bearing names, as they will be relabelled ===


,complaints,original,type,canonical_label,predecessor_named
0,266,Ningo Lending LLC dba Spotloan,DBA,Ningo Lending LLC,
1,99,"Uetsa Tsakits, Inc. d/b/a MaxLend",DBA,"Uetsa Tsakits, Inc.",
2,98,"Populus Financial Group, Inc. (F/K/A Ace Cash Express)",FKA,"Populus Financial Group, Inc.",(F/K/A Ace Cash Express)
3,97,Minto Financial dba Minto Money,DBA,Minto Financial,
4,44,MoneySpot USA LLC DBA Sunshine Loans,DBA,MoneySpot USA LLC,
5,35,"Makes Cents, Inc. dba Advance.Cash",DBA,"Makes Cents, Inc.",
6,21,Florida Financial Management dba Lend A Dollar,DBA,Florida Financial Management,
7,20,Cliff Lending Inc. d/b/a Advance Loan Solutions,DBA,Cliff Lending Inc.,
8,17,FinCo Services Inc DBA Current,DBA,FinCo Services Inc,
9,16,Bear Claw Lending dba Lucent Cash,DBA,Bear Claw Lending,


## 5. Fuzzy matching — candidates only, never merges

Two independent candidate generators, both reported and both left unmerged:
a token-multiset near-duplicate test (catches what character similarity misses) and
`difflib` ratio above 0.90.

In [8]:
tok = {n: Counter([t for t in stem(n).split() if t not in SUFFIXES] or stem(n).split())
       for n in names}

tok_pairs = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = tok[names[i]], tok[names[j]]
        d = sum(((a - b) + (b - a)).values())
        if 0 < d <= 1 and a != b:
            tok_pairs.append((names[i], names[j], d))
tok_pairs.sort(key=lambda p: -(counts[p[0]] + counts[p[1]]))

print(f"=== token-multiset near-duplicates (<=1 token apart): {len(tok_pairs)} ===")
print(f"{'n1':>6} {'n2':>6}  names")
for a, b, d in tok_pairs:
    print(f"{counts[a]:>6} {counts[b]:>6}  {a}   ||   {b}")

=== token-multiset near-duplicates (<=1 token apart): 9 ===
    n1     n2  names
    95      2  1st Franklin Financial Corporation   ||   Franklin Financial  Corporation
    36      7  CREDIT ACCEPTANCE CORPORATION   ||   American Credit Acceptance, LLC
    19      1  National Financial LLC   ||   Security National Financial Corp
    17      1  Debt Recovery Solutions, LLC   ||   Debt Solutions LLC
     3      1  Better Debt Solutions LLC   ||   Debt Solutions LLC
     2      1  First Credit Corporation   ||   First Financial Credit, Inc
     2      1  Advantage One Credit, LLC   ||   Credit One LLC
     1      1  Access Loan Services Inc.   ||   ACCESS LOAN COMPANY
     1      1  Consumer Financial Services Solutions, Inc.   ||   Consumer Financial Services


In [9]:
block = defaultdict(list)
for n in names:
    for t in set(stem(n).split()):
        if len(t) >= 4:
            block[t].append(n)

fuzzy, seen = [], set()
for grp in block.values():
    for i in range(len(grp)):
        for j in range(i + 1, len(grp)):
            a, b = sorted((grp[i], grp[j]))
            if (a, b) in seen:
                continue
            seen.add((a, b))
            r = difflib.SequenceMatcher(None, stem(a), stem(b)).ratio()
            if r >= 0.90:
                fuzzy.append((r, a, b))
fuzzy.sort(reverse=True)

print(f"=== difflib candidates >= 0.90: {len(fuzzy)} ===")
print(f"{'ratio':>6} {'n1':>6} {'n2':>6}  pair")
for r, a, b in fuzzy:
    print(f"{r:>6.3f} {counts[a]:>6} {counts[b]:>6}  {a}   ||   {b}")

=== difflib candidates >= 0.90: 20 ===
 ratio     n1     n2  pair
 1.000      8      9  Time Investment Company, Inc.   ||   Time Investment Corporation
 1.000     51     24  FIRST TECHNOLOGY FEDERAL CREDIT UNION   ||   First Technology Federal Credit Union
 0.947      3      1  FC HoldCo LLC   ||   FSC HOLDCO, LLC
 0.929      3      3  Onset Financial, Inc.   ||   SET Financial Corporation
 0.923      6      2  HFS Financial LLC   ||   TFS FINANCIAL CORPORATION
 0.923      1    219  CIG FINANCIAL LLC   ||   CNG FINANCIAL CORPORATION
 0.923      4      1  CCI Financial, Inc   ||   CIG FINANCIAL LLC
 0.923      1      2  Alliance Financial Services, LLC   ||   Allied Financial Services Inc
 0.919      1      1  BCI FINANCIAL GROUP, INC.   ||   BH Financial Group, LLC
 0.918    110      1  BMO BANK NATIONAL ASSOCIATION   ||   BREMER BANK, NATIONAL ASSOCIATION
 0.917     28      1  Eagle Financial Services, Inc.   ||   Elite Financial Services, Inc.
 0.914      1      5  DSR HOLDINGS GROU

**Every one of these is a false positive, and that is the point of inspecting them.**
`BREMER BANK` is not `BMO BANK`; `CURO Intermediate Holdings` is not `CCF Intermediate
Holdings`; `Elite Financial Services` is not `Eagle Financial Services`;
`1st Franklin Financial` is not `Franklin Financial`. A generic
`[ADJECTIVE] FINANCIAL SERVICES` naming convention produces a large number of
high-similarity pairs between genuinely unrelated lenders. None are merged.

## 6. Name changes surfaced by the narrative join

The narrative-coverage notebook joined our extract to the FOIA archive on `Complaint ID`
and found 284 complaints where the archive reports a different company name. Those are
not duplicates inside our extract — they are a **staleness signal**: the archive is a
later snapshot than our CSV.

In [10]:
import os

if os.path.exists(NARR):
    narr = pd.read_csv(NARR, usecols=["Complaint ID", "Company"]).rename(
        columns={"Company": "archive_company"})
    j = df.merge(narr, on="Complaint ID", how="inner")
    dis = j[j["Company"] != j["archive_company"]]
    print(f"matched complaints: {len(j):,}")
    print(f"rows where the archive reports a different name: {len(dis):,}\n")
    out = (dis.groupby(["Company", "archive_company"])
              .agg(complaints=("Complaint ID", "size"),
                   our_first=("Date received", "min"),
                   our_last=("Date received", "max"))
              .reset_index())
    out["successor_in_our_extract"] = [
        name in set(df["Company"]) for name in out["archive_company"]]
    out
else:
    print("narrative cache not present; run notebooks/cfpb_narrative_coverage.ipynb first")

matched complaints: 37,322
rows where the archive reports a different name: 284



In [11]:
# The decisive check. If a successor name is not itself a row in our extract then there
# is nothing to merge: the two spellings never co-occur, so collapsing them would be a
# no-op at best and a fabrication at worst.
ours_set = set(df["Company"])
for succ in ["BlueChip Financial", "STATE EMPLOYEES CREDIT UNION", "LAKEVIEW LENDING LLC"]:
    print(f"{succ!r:36} present in our extract: {succ in ours_set}")

'BlueChip Financial'                 present in our extract: False
'STATE EMPLOYEES CREDIT UNION'       present in our extract: False
'LAKEVIEW LENDING LLC'               present in our extract: False


All three successors are **absent** from our extract, so all three are recorded as
`ownership_change` with `merge = False`. The one case needing a judgement call is
`American First Funding LLC` -> `LAKEVIEW LENDING LLC`: our extract separately contains
`LAKEVIEW LOAN SERVICING, LLC` (1 complaint), a different legal name. Linking them would
be an assumption, so it is filed as `manual_review`.

Note also that `Ningo Lending LLC dba Spotloan` spans 2023-09 to 2026-08 under a single
row while the archive calls the same complaint IDs `BlueChip Financial` from partway
through. The rename happened *inside* our observation window, so any time series for
that filer crosses an ownership change.

## 7. Build the mapping

Decision rules, in the order they are applied. Each one is one-sided.

| Rule | match_type | merge? |
| --- | --- | --- |
| identical `alnum` key as another name | `formatting` | yes |
| identical `stem` key, different `alnum` | `manual_review` | **no** — needs human judgement |
| carries a `(F/K/A ...)` / `(formerly ...)` parenthetical | `fka` | **no** — names a *predecessor* |
| carries a trailing `d/b/a` brand | `dba` | **no** — a brand is not a second filer |
| archive reports a successor name for the same `Complaint ID` | `ownership_change` | **no** — successor absent from our extract |
| surfaced by fuzzy / token candidate generation, unproven | `manual_review` | **no** |
| nothing above | `exact` | n/a |

`canonical_company` is the most frequent surface form in the group, DBA/FKA markers
removed. `canonical_key` is the `stem` key, provided so downstream grouping never has to
re-derive it.

In [12]:
RECORDED_CHANGES = {
    "Ningo Lending LLC dba Spotloan": (
        "BlueChip Financial",
        "archive reports the successor name for the same Complaint IDs from partway "
        "through our window; successor absent from our extract so nothing to merge",
    ),
    "American First Funding LLC": (
        "LAKEVIEW LENDING LLC",
        "archive reports successor; our extract separately holds LAKEVIEW LOAN "
        "SERVICING, LLC (1 complaint) - possible but unproven link, not merged",
    ),
    "STATE EMPLOYEES’ CREDIT UNION": (
        "STATE EMPLOYEES CREDIT UNION",
        "archive drops the typographic apostrophe; same filer, spelling difference only",
    ),
}

# Unproven candidates: name -> the other name it might be, kept for human review.
AMBIGUOUS = {
    "Time Investment Corporation": "Time Investment Company, Inc.",
    "Time Investment Company, Inc.": "Time Investment Corporation",
    "ACCESS LOAN COMPANY": "Access Loan Services Inc.",
    "Access Loan Services Inc.": "ACCESS LOAN COMPANY",
    "Consumer Financial Services": "Consumer Financial Services Solutions, Inc.",
    "Consumer Financial Services Solutions, Inc.": "Consumer Financial Services",
}

rows = []
for n in names:
    al, st = alnum(n), stem(n)
    group = alnum_map[al]
    note, cand, successor = "", "", ""

    if len(group) > 1:
        canonical = max(group, key=lambda z: (counts[z], z))
        mtype = "formatting"
        # merge=True only when this name actually collapses onto a different label.
        # The group's surviving member is left as merge=False.
        merge = canonical != n
        others = [z for z in group if z != n]
        note = ("identical ignoring case/punctuation/whitespace; merged with "
                + "; ".join(sorted(others)))
    elif len(stem_map[st]) > 1:
        canonical = n
        mtype, merge = "manual_review", False
        cand = "; ".join(z for z in stem_map[st] if z != n)
        note = (f"same stem after legal-suffix removal as {cand}, but different "
                "geographic footprint - not merged")
    elif n in RECORDED_CHANGES:
        # Checked before the d/b/a branch: an ownership change is the more specific
        # and more actionable fact, and must not be masked by "this has a brand".
        successor, why = RECORDED_CHANGES[n]
        canonical, mtype, merge = strip_marker(n), "ownership_change", False
        note = f"successor name {successor!r}: {why}"
        if canonical != n:
            note += f". Label also carries a d/b/a brand; canonical label is {canonical!r}"
    elif HISTORY_PAREN.search(n):
        canonical, mtype, merge = strip_marker(n), "fka", False
        note = ("parenthetical names a predecessor entity, not this filer; removed from "
                "the label. Predecessor: " + "; ".join(HISTORY_PAREN.findall(n)))
    elif TRAILING_MARKER.search(n):
        canonical, mtype, merge = strip_marker(n), "dba", False
        note = ("d/b/a brand is the same filer trading under a brand; canonical label is "
                "the legal entity. Brand: "
                + re.split(MARKER_ONLY, n)[-1].strip(" ,;()-"))
    elif n in AMBIGUOUS:
        canonical, mtype, merge = n, "manual_review", False
        cand = AMBIGUOUS[n]
        note = f"candidate duplicate of {cand!r}; similarity only, not merged"
    else:
        canonical, mtype, merge = n, "exact", False
        note = ""

    rows.append({
        "original_company": n,
        "canonical_company": canonical,
        "canonical_key": stem(canonical),
        "match_type": mtype,
        "merge": merge,
        "complaints": int(counts[n]),
        "successor_name": successor,
        "candidate_duplicate_of": cand,
        "notes": note,
    })

aliases = pd.DataFrame(rows).sort_values(["canonical_company", "original_company"]).reset_index(drop=True)
print(f"rows written: {len(aliases):,} (one per distinct original name)")
print(f"labels changed : {(aliases.original_company != aliases.canonical_company).sum():,}")
print(f"actual merges  : {int(aliases["merge"].sum()):,} names collapsed into another")
aliases.head(8)

rows written: 1,162 (one per distinct original name)
labels changed : 34
actual merges  : 1 names collapsed into another


,original_company,canonical_company,canonical_key,match_type,merge,complaints,successor_name,candidate_duplicate_of,notes
0,"1 STOP MONEY CENTERS, LLC","1 STOP MONEY CENTERS, LLC",1 STOP MONEY CENTERS,exact,False,1,,,
1,1 Stop Payday Loan Services LLC,1 Stop Payday Loan Services LLC,1 STOP PAYDAY LOAN SERVICES,exact,False,1,,,
2,1st Franklin Financial Corporation,1st Franklin Financial Corporation,1ST FRANKLIN FINANCIAL,exact,False,95,,,
3,1st Nationwide Mortgage of Texas Corp.,1st Nationwide Mortgage of Texas Corp.,1ST NATIONWIDE MORTGAGE OF TEXAS,exact,False,4,,,
4,1st Step Financial INC.,1st Step Financial INC.,1ST STEP FINANCIAL,exact,False,1,,,
5,2233 Paradise Road LLC,2233 Paradise Road LLC,2233 PARADISE ROAD,exact,False,6,,,
6,2288984 Ontario Inc.,2288984 Ontario Inc.,2288984 ONTARIO,exact,False,123,,,
7,"3D Financial, LP","3D Financial, LP",3D FINANCIAL,exact,False,1,,,


The `d/b/a` and `F/K/A` rows change a label without merging anything, so a `merge = True`
flag is what actually reduces the lender count.

In [13]:
tmp = f"{OUT}.tmp"
aliases.to_csv(tmp, index=False)
import os
os.replace(tmp, OUT)
print(f"wrote {OUT}")
print(f"size: {os.path.getsize(OUT):,} bytes")
print("\nverify it round-trips:")
chk = pd.read_csv(OUT)
print(f"  rows={len(chk):,}  cols={list(chk.columns)}")
print(f"  every original_company resolvable: {set(chk.original_company) == set(names)}")

wrote /Users/mihirkale/cdc/data/processed/company_aliases.csv
size: 107,664 bytes

verify it round-trips:
  rows=1,162  cols=['original_company', 'canonical_company', 'canonical_key', 'match_type', 'merge', 'complaints', 'successor_name', 'candidate_duplicate_of', 'notes']
  every original_company resolvable: True


## 8. Effect on the lender count

In [14]:
canonical_names = aliases["canonical_company"].nunique()
print(f"unique Company values before : {len(names):,}")
print(f"unique Company values after  : {canonical_names:,}")
print(f"net reduction                : {len(names) - canonical_names:,}")

print(f"\n=== counts by match_type ===")
mt = (aliases.groupby("match_type")
             .agg(names=("original_company", "size"),
                  complaints=("complaints", "sum"),
                  relabelled=("original_company",
                              lambda s: (s != aliases.loc[s.index, "canonical_company"]).sum()))
             .sort_values("names", ascending=False))
mt

unique Company values before : 1,162
unique Company values after  : 1,161
net reduction                : 1

=== counts by match_type ===


,names,complaints,relabelled
match_type,,,
exact,1119,37459,0
dba,30,419,30
manual_review,6,21,0
ownership_change,3,302,1
fka,2,99,2
formatting,2,75,1


In [15]:
print("=== names that actually merge (merge = True) ===")
m = aliases[aliases["merge"]]
for _, r in m.iterrows():
    sibs = aliases[(aliases.canonical_company == r.canonical_company) & (aliases["merge"])]
    print(f"\n  {r.canonical_company!r}")
    for _, s in sibs.iterrows():
        print(f"      {s.complaints:>6}  {s.original_company!r}  [{s.match_type}]")
    print(f"      -> {sibs.complaints.sum():,} complaints under {r.canonical_company!r}")

=== names that actually merge (merge = True) ===

  'FIRST TECHNOLOGY FEDERAL CREDIT UNION'
          24  'First Technology Federal Credit Union'  [formatting]
      -> 24 complaints under 'FIRST TECHNOLOGY FEDERAL CREDIT UNION'


In [16]:
print("=== relabels that do NOT merge (largest 15 by complaints) ===")
relab = aliases[(aliases.original_company != aliases.canonical_company) & (~aliases["merge"])]
relab[["complaints", "original_company", "canonical_company", "match_type"]].head(15)

=== relabels that do NOT merge (largest 15 by complaints) ===


,complaints,original_company,canonical_company,match_type
108,1,Atlantic Discount Corporation dba Atlantic Financial Services,Atlantic Discount Corporation,dba
154,13,"BYS, LLC. d/b/a Boost Your Score","BYS, LLC.",dba
156,16,Bear Claw Lending dba Lucent Cash,Bear Claw Lending,dba
174,3,"Blue Front Financial, LLC d/b/a Ladder Loans","Blue Front Financial, LLC",dba
280,20,Cliff Lending Inc. d/b/a Advance Loan Solutions,Cliff Lending Inc.,dba
308,1,"Continental Services Group, Inc. d/b/a ConServe","Continental Services Group, Inc.",dba
357,4,Double D Finance dba Idaho Finance,Double D Finance,dba
442,17,FinCo Services Inc DBA Current,FinCo Services Inc,dba
465,21,Florida Financial Management dba Lend A Dollar,Florida Financial Management,dba
518,2,"Good News Group, LLC d/b/a Savvy Loans","Good News Group, LLC",dba


In [17]:
print("=== ambiguous cases left unresolved ===")
amb = aliases[aliases.match_type == "manual_review"]
print(f"{len(amb):,} names flagged\n")
amb[["complaints", "original_company", "canonical_company", "candidate_duplicate_of", "notes"]].to_string(index=False)

=== ambiguous cases left unresolved ===
6 names flagged



" complaints                            original_company                           canonical_company                      candidate_duplicate_of                                                                                                                  notes\n          1                         ACCESS LOAN COMPANY                         ACCESS LOAN COMPANY                   Access Loan Services Inc.                                        candidate duplicate of 'Access Loan Services Inc.'; similarity only, not merged\n          1                   Access Loan Services Inc.                   Access Loan Services Inc.                         ACCESS LOAN COMPANY                                              candidate duplicate of 'ACCESS LOAN COMPANY'; similarity only, not merged\n          1                 Consumer Financial Services                 Consumer Financial Services Consumer Financial Services Solutions, Inc.                      candidate duplicate of 'Consumer Financial

## 9. Company x Sub-product before vs. after

This is the question that decides whether the mapping is worth anything downstream.

In [18]:
df["canonical_company"] = df["Company"].map(
    aliases.set_index("original_company")["canonical_company"])

before = df.groupby(["Company", "Sub-product"]).size().rename("complaints_before")
after = df.groupby(["canonical_company", "Sub-product"]).size().rename("complaints_after")
relabelled = set(aliases.loc[aliases.original_company != aliases.canonical_company, "original_company"])

print(f"Company x Sub-product groups before : {len(before):,}")
print(f"Company x Sub-product groups after  : {len(after):,}")
print(f"net change                         : {len(after) - len(before):+,}")
print(f"\ntotal complaints                    : {before.sum():,} -> {after.sum():,}")

Company x Sub-product groups before : 2,419
Company x Sub-product groups after  : 2,415
net change                         : -4

total complaints                    : 38,375 -> 38,375


In [19]:
# Classify what actually happened to each post-canonicalization group.
#
# Two very different things get conflated in a naive before/after: a group whose
# *label* changed (a d/b/a brand removed) and a group that actually *gained or lost*
# complaints (two names collapsed into one). Keying the comparison on the canonical
# label makes every rename look like a size change, which is an artifact. So compare
# the number of distinct source names feeding each group instead.
g = (df.groupby(["canonical_company", "Sub-product"])
        .agg(complaints_after=("Complaint ID", "size"),
             n_source_names=("Company", "nunique"),
             sole_source=("Company", "first"))
        .reset_index())

g["status"] = "untouched"
g.loc[g.sole_source != g.canonical_company, "status"] = "relabelled_only"
g.loc[g.n_source_names > 1, "status"] = "merged_members"

before_sizes = df.groupby(["Company", "Sub-product"]).size()

# A relabelled group kept exactly its contents, so its "before" size must be looked up
# under the name it had *before* canonicalization, not under the new label.
prior_label = g["canonical_company"].where(g["status"] != "relabelled_only", g["sole_source"])
g["complaints_before"] = [int(before_sizes.get((c, s), 0))
                          for c, s in zip(prior_label, g["Sub-product"])]
g["delta"] = g["complaints_after"] - g["complaints_before"]

print(f"Company x Sub-product groups before : {len(before_sizes):,}")
print(f"Company x Sub-product groups after  : {len(g):,}")
print(f"net change                          : {len(g) - len(before_sizes):+,}")
print(f"total complaints                    : {g['complaints_after'].sum():,}")
print()
print(g["status"].value_counts().to_frame("groups"))

Company x Sub-product groups before : 2,419
Company x Sub-product groups after  : 2,415
net change                          : -4
total complaints                    : 38,375

                 groups
status                 
untouched          2340
relabelled_only      71
merged_members        4


In [20]:
print("=== groups that genuinely gained or lost complaints (real merges) ===")
ch = g[g["status"] == "merged_members"].sort_values("delta", key=abs, ascending=False)
print(ch[["canonical_company", "Sub-product", "n_source_names",
          "complaints_before", "complaints_after", "delta"]].to_string(index=False))
print(f"\ncomplaints absorbed by real merges : {int(ch['delta'].sum()):,}")

print("\n=== groups only relabelled: contents identical, delta 0 ===")
rel = g[g["status"] == "relabelled_only"]
print(f"groups   : {len(rel):,}")
print(f"complaints: {int(rel['complaints_after'].sum()):,}")
print(f"every one has delta == 0 : {bool((rel['delta'] == 0).all())}")

=== groups that genuinely gained or lost complaints (real merges) ===
                    canonical_company             Sub-product  n_source_names  complaints_before  complaints_after  delta
FIRST TECHNOLOGY FEDERAL CREDIT UNION        Installment loan               2                 39                53     14
FIRST TECHNOLOGY FEDERAL CREDIT UNION Personal line of credit               2                 10                17      7
FIRST TECHNOLOGY FEDERAL CREDIT UNION             Payday loan               2                  1                 3      2
FIRST TECHNOLOGY FEDERAL CREDIT UNION              Title loan               2                  1                 2      1

complaints absorbed by real merges : 24

=== groups only relabelled: contents identical, delta 0 ===
groups   : 71
complaints: 784
every one has delta == 0 : True


In [21]:
print("=== materiality summary ===")
merged = g[g["status"] == "merged_members"]
relab = g[g["status"] == "relabelled_only"]
print(f"""
lender names before / after             : {len(names):,} / {aliases["canonical_company"].nunique():,}
lender x sub-product groups before/after: {len(before_sizes):,} / {len(g):,}
groups that gained or lost complaints   : {len(merged):,}
largest single-group change             : {int(merged["delta"].abs().max()) if len(merged) else 0} complaints
complaints absorbed by real merges      : {int(merged["delta"].sum()):,}  ({merged["delta"].sum() / len(df) * 100:.3f}% of the dataset)
groups relabelled only                  : {len(relab):,}  (contents identical, delta 0)
groups fully untouched                  : {int((g["status"] == "untouched").sum()):,}
lender names relabelled                 : {int((aliases.original_company != aliases.canonical_company).sum()):,}
lender names actually merged            : {int(aliases["merge"].sum()):,}
""")

=== materiality summary ===

lender names before / after             : 1,162 / 1,161
lender x sub-product groups before/after: 2,419 / 2,415
groups that gained or lost complaints   : 4
largest single-group change             : 14 complaints
complaints absorbed by real merges      : 24  (0.063% of the dataset)
groups relabelled only                  : 71  (contents identical, delta 0)
groups fully untouched                  : 2,340
lender names relabelled                 : 34
lender names actually merged            : 1



## Conclusion — does company identity normalization materially affect the lender x sub-product analysis?

**No. Not for this dataset.** The mapping is still worth committing as documentation and as
insurance against a future extract, but it changes no result worth reporting.

The reason is simply how little duplication exists. Across 1,162 company names and 38,375
complaints there is exactly **one** true duplicate: `First Technology Federal Credit Union`
in two capitalizations (51 and 24 complaints), merged into one.

| | before | after |
| --- | --- | --- |
| distinct lender names | 1,162 | 1,161 |
| Company x Sub-product groups | 2,419 | 2,415 |
| groups that gained or lost complaints | — | 4 |
| groups relabelled only (contents identical) | — | 71 |
| groups fully untouched | — | 2,340 |

The entire measurable effect is **24 complaints, 0.063% of the dataset**, all of it
`First Technology Federal Credit Union` absorbing its own lower-case twin across four
sub-products (largest single group: Installment loan, 39 -> 53). Nothing else moves.

### What the mapping does deliver

`data/processed/company_aliases.csv` holds one row per original name (1,162 rows, 9
columns). Distribution by `match_type`:

| match_type | names | complaints | relabelled |
| --- | --- | --- | --- |
| `exact` | 1,119 | 37,459 | 0 |
| `dba` | 30 | 419 | 30 |
| `manual_review` | 6 | 21 | 0 |
| `ownership_change` | 3 | 302 | 1 |
| `fka` | 2 | 99 | 2 |
| `formatting` | 2 | 75 | 1 |

- **32 `dba` / `fka` relabels.** A brand is not a second filer, so
  `Ningo Lending LLC dba Spotloan` canonically becomes `Ningo Lending LLC`. These change
  labels without changing any group's contents (delta 0 across 71 groups). Join keys are
  unaffected either way.
- **3 documented name changes** recovered from the narrative join, **none mergeable**:
  `Ningo Lending LLC dba Spotloan` -> `BlueChip Financial` (254 complaints),
  `American First Funding LLC` -> `LAKEVIEW LENDING LLC` (1),
  `STATE EMPLOYEES' CREDIT UNION` -> apostrophe-free spelling (29). In all three the
  successor is **absent from our extract**, so there is nothing to collapse. They are kept
  as `ownership_change` with a `successor_name` column precisely so they are
  distinguishable from a plain alias.
- **6 names left in `manual_review`** with the competing candidate recorded in
  `candidate_duplicate_of`, for a human to resolve.

### Which mapping downstream should use

Join on `original_company`; **group by `canonical_key`**. `canonical_key` is the
conservative machine key (case and punctuation folded, legal suffixes dropped).
`canonical_company` is the human-readable label and must **not** be used as a join target,
because 34 rows deliberately differ from their original spelling. `merge = True` is the
only flag that indicates two names actually collapsed into one, and exactly one row carries it.

### Three cautions that survive this exercise

1. **This is a property of the current extract, not of CFPB data.** Our CSV is one filtered
   snapshot that happened to contain one row per filer spelling. A refreshed or differently
   filtered extract could reintroduce exactly the variants the deleted `normalize.py` was
   written for — its docstring cites
   `Elevate Financial, LLC (F/K/A Enova International)`, which does not appear here at all.
   That normalizer and its suffix/generic-word rules remain worth recovering; they are just
   not warranted by *this* file.
2. **The extract is a stale snapshot, and this is now measurable.** The archive reports newer
   names for 284 of our complaints, and `Ningo Lending LLC dba Spotloan` spans 2023-09 to
   2026-08 under a single row while being renamed mid-window. Any lender time series built on
   this CSV crosses at least one ownership change.
3. **The candidate lists are traps, not leads.** 20 `difflib` pairs at or above 0.90 and 9
   token-multiset near-duplicates were inspected; every one is an unrelated pair —
   `BREMER BANK` / `BMO BANK NATIONAL ASSOCIATION`, `CURO Intermediate Holdings` /
   `CCF Intermediate Holdings LLC`, `Elite` / `Eagle Financial Services`,
   `Halsted` / `Allied Financial Services`. The `[ADJECTIVE] FINANCIAL SERVICES` naming
   convention guarantees a steady supply of such pairs. Any future automated matcher will
   generate them, and merging on similarity alone would corrupt the lender set.

The `Company` field here is, in practical terms, already clean. Effort is better spent on
the missing denominator than on name resolution.